# Notebook 07 -- Optimización de Hiperparámetros

---


## Objetivo

Optimizar hiperparámetros de modelos expertos y gating mediante Random Search + Successive Halving. Generar curvas de convergencia y tabla de mejores parámetros por experto.


## Entrada

features + modelos expertos + gating mechanism


## Salida

Hiperparámetros optimizados (JSON) + modelos re-entrenados con mejores parámetros


## Instalación

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

## Configuración

In [ ]:
import os, json, warnings, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.stats import uniform, loguniform, randint

from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import (
    HalvingRandomSearchCV,
    TimeSeriesSplit,
)
from sklearn.linear_model import Ridge, ElasticNet, Lasso, BayesianRidge, SGDRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # Added import for SimpleImputer

import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR = "../data"
MODELS_DIR = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
RANDOM_SEED = 42
N_JOBS      = -1

for d in [MODELS_DIR, RESULTS_DIR, f"{RESULTS_DIR}/figures"]:
    os.makedirs(d, exist_ok=True)

# ── Configuración de gráficos de calidad de publicación ─────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("colorblind")

np.random.seed(RANDOM_SEED)
print(f"Configuración cargada — seed={RANDOM_SEED}, n_jobs={N_JOBS}")

## Cargar artefactos de notebooks anteriores

Cargamos tres artefactos:

| Artefacto | Fuente | Descripción |
|-----------|--------|-------------|
| `features_df` | Notebook 03 | Matriz de variables horarias con todas las variables generadas |
| `cluster_assignments` | Notebook 04 | Mapeo `date → cluster_id` para cada hora |
| `best_models` | Notebook 05 | Diccionario `expert_id → model_family` (mejor modelo por clúster) |

In [ ]:
# ── 3.1  Matriz de variables ──────────────────────────────────────────────────
features_path = Path(DATA_DIR) / "processed" / "features.parquet"
if features_path.exists():
    features_df = pd.read_parquet(features_path)
    print(f"Variables cargadas: {features_df.shape}")
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {features_path}. Ejecuta antes el notebook 02 (make pipeline).")

In [ ]:
features_df = features_df[~features_df.index.year.isin([2020, 2021])]

In [ ]:
# ── 3.2  Asignaciones de clúster ─────────────────────────────────────────────
cluster_path = Path(DATA_DIR) / "clustering" / "cluster_assignments.parquet"
if cluster_path.exists():
    cluster_df = pd.read_parquet(cluster_path)
    print(f"Asignaciones de clúster cargadas: {cluster_df.shape}")
    # Asegurar que la columna 'datetime' esté como índice y nombrada 'fecha' para consistencia con features_df
    if "datetime" in cluster_df.columns:
        cluster_df.set_index("datetime", inplace=True)
        cluster_df.index.name = "fecha" # Estandarizar nombre del índice

    if "expert_id" in cluster_df.columns:
        # Mapear expert_ids de texto a cluster_ids numéricos para procesamiento interno y ordenamiento
        unique_expert_ids = sorted(cluster_df["expert_id"].unique())
        expert_id_to_cluster_id = {eid: i for i, eid in enumerate(unique_expert_ids)}
        cluster_df["cluster_id"] = cluster_df["expert_id"].map(expert_id_to_cluster_id)
        # Guardar mapeo inverso para posible depuración/visualización
        cluster_id_to_expert_id = {i: eid for i, eid in enumerate(unique_expert_ids)}
        print(f"expert_ids de texto mapeados a cluster_ids numéricos: {cluster_id_to_expert_id}")
    elif "cluster_id" in cluster_df.columns:
        print("Usando la columna 'cluster_id' existente del parquet.")
    else:
        raise ValueError("No se encontró una columna válida de identificador de clúster ('expert_id' o 'cluster_id') en cluster_df cargado.")

else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {cluster_path}. Ejecuta antes el notebook 03 (make pipeline).")


# Cada asignación trae su fecha (columna datetime, ver notebook 03). Reconstruirla
# por posición la desplazaba 48 h, porque el clustering descarta la primera semana ISO incompleta.
cluster_df['timestamp'] = cluster_df.index
cluster_df = cluster_df.reset_index(drop=True)

print(f"\nForma de los clústeres: {cluster_df.shape}")
print(f"Columnas de los clústeres: {list(cluster_df.columns)}")

# Asegurar que timestamp sea datetime en ambos
cluster_df['timestamp'] = pd.to_datetime(cluster_df['timestamp'])

cluster_df.index = cluster_df['timestamp']
cluster_df.drop(columns=['timestamp'], inplace=True)

In [ ]:
cluster_df.head()

In [ ]:
# ── 3.3  Selección del mejor modelo por experto ─────────────────────────────────
best_models_path = Path(RESULTS_DIR) / "05_expert_selection_best.csv"
os.makedirs(best_models_path.parent, exist_ok=True)

if best_models_path.exists():
    df_aux = pd.read_csv(best_models_path)
    best_models = dict(zip(df_aux.index, df_aux.model))

    print(f"Mejores modelos cargados: {len(best_models)} expertos")
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {best_models_path}. Ejecuta antes el notebook 05 (make pipeline).")

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(features_df.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
# ── 3.4  Unir variables + cluster_id ─────────────────────────────────────
# features_df ya tiene 'fecha' (o 'datetime') como índice desde la carga inicial/generación sintética.
# cluster_df también tiene 'fecha' (o 'datetime') como índice.
if "cluster_id" not in features_df.columns:
    features_df = features_df.join(cluster_df[["cluster_id"]], how="inner")

TARGET = "demand"
#FEATURE_COLS = [c for c in features_df.columns if c not in [TARGET, "cluster_id"]]
FEATURE_COLS = calendar_lags

print(f"\nDataset final: {features_df.shape[0]:,} filas, {len(FEATURE_COLS)} variables")
print(f"Expertos (clústeres): {sorted(features_df['cluster_id'].unique())}")
print(f"Asignaciones de modelos: {best_models}")

---

## Definir Espacios de Búsqueda de Hiperparámetros por Familia de Modelos

Cada familia de modelos tiene un espacio de búsqueda adaptado. Las distribuciones se eligen como
log-uniformes para parámetros de escala (learning rate, regularización) y uniformes o
uniformes discretas para parámetros estructurales (profundidad, número de hojas).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Definición de espacios de búsqueda
# ══════════════════════════════════════════════════════════════════════════

SEARCH_SPACES: dict[str, dict] = {
    # ── Linear family ────────────────────────────────────────────────────
    "ridge": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")), # Imputador agregado

            ("scaler", StandardScaler()),
            ("reg", Ridge(random_state=RANDOM_SEED)),
        ]),
        "params": {
            "reg__alpha": loguniform(1e-3, 1e3),
        },
    },
    "lasso": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")), # Imputador agregado

            ("scaler", StandardScaler()),
            ("reg", Lasso(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha": loguniform(1e-4, 1e2),
        },
    },
    "elasticnet": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")), # Imputador agregado

            ("scaler", StandardScaler()),
            ("reg", ElasticNet(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha":    loguniform(1e-4, 1e2),
            "reg__l1_ratio": uniform(0.01, 0.98),  # [0.01, 0.99]
        },
    },
    "bayesian_ridge": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")), # Imputador agregado

            ("scaler", StandardScaler()),
            ("reg", BayesianRidge()),
        ]),
        "params": {
            "reg__alpha_1":  loguniform(1e-7, 1e-1),
            "reg__alpha_2":  loguniform(1e-7, 1e-1),
            "reg__lambda_1": loguniform(1e-7, 1e-1),
            "reg__lambda_2": loguniform(1e-7, 1e-1),
        },
    },
    "sgd": {
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="mean")), # Imputador agregado

            ("scaler", StandardScaler()),
            ("reg", SGDRegressor(random_state=RANDOM_SEED, max_iter=5000)),
        ]),
        "params": {
            "reg__alpha":    loguniform(1e-6, 1e-1),
            "reg__l1_ratio": uniform(0.0, 1.0),
            "reg__penalty":  ["l2", "l1", "elasticnet"],
        },
    },

    # ── Boosting family ──────────────────────────────────────────────────
    "lgbm": {
        "model": lgb.LGBMRegressor(
            random_state=RANDOM_SEED, n_jobs=1, verbosity=-1,
        ),
        "params": {
            "learning_rate":     loguniform(1e-3, 3e-1),
            "n_estimators":      randint(100, 1500),
            "max_depth":         randint(3, 12),
            "num_leaves":        randint(15, 255),
            "min_child_samples": randint(5, 100),
            "subsample":         uniform(0.5, 0.5),        # [0.5, 1.0]
            "colsample_bytree":  uniform(0.4, 0.6),        # [0.4, 1.0]
            "reg_alpha":         loguniform(1e-4, 1e1),
            "reg_lambda":        loguniform(1e-4, 1e1),
        },
    },
    "xgboost": {
        "model": xgb.XGBRegressor(
            random_state=RANDOM_SEED, n_jobs=1, verbosity=0,
            tree_method="hist",
        ),
        "params": {
            "learning_rate":     loguniform(1e-3, 3e-1),
            "n_estimators":      randint(100, 1500),
            "max_depth":         randint(3, 12),
            "min_child_weight":  randint(1, 30),
            "subsample":         uniform(0.5, 0.5),
            "colsample_bytree":  uniform(0.4, 0.6),
            "reg_alpha":         loguniform(1e-4, 1e1),
            "reg_lambda":        loguniform(1e-4, 1e1),
            "gamma":             loguniform(1e-4, 1e1),
        },
    },
    "gradient_boosting": {
        "model": GradientBoostingRegressor(random_state=RANDOM_SEED),
        "params": {
            "learning_rate":    loguniform(1e-3, 3e-1),
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(3, 10),
            "min_samples_split":randint(2, 50),
            "min_samples_leaf": randint(1, 30),
            "subsample":        uniform(0.5, 0.5),
            "max_features":     uniform(0.3, 0.7),
        },
    },

    # ── Forest family ────────────────────────────────────────────────────
    "random_forest": {
        "model": RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=1),
        "params": {
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(5, 30),
            "min_samples_split":randint(2, 30),
            "min_samples_leaf": randint(1, 20),
            "max_features":     uniform(0.3, 0.7),
        },
    },
    "extra_trees": {
        "model": ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=1),
        "params": {
            "n_estimators":     randint(100, 1000),
            "max_depth":        randint(5, 30),
            "min_samples_split":randint(2, 30),
            "min_samples_leaf": randint(1, 20),
            "max_features":     uniform(0.3, 0.7),
        },
    },
}

print("Espacios de búsqueda definidos para:")
for name, spec in SEARCH_SPACES.items():
    n_hp = len(spec["params"])
    print(f"  {name:22s}  {n_hp} hiperparámetros")

In [ ]:
param_specs = []
models_to_export = ["bayesian_ridge", "lasso", "lgbm"]

for model_name in models_to_export:
    if model_name in SEARCH_SPACES:
        params = SEARCH_SPACES[model_name]["params"]
        for param_name, distribution in params.items():
            entry = {"model": model_name, "parameter": param_name}

            # Check if it's a scipy.stats frozen distribution
            if hasattr(distribution, 'dist') and hasattr(distribution.dist, 'name'):
                dist_name = distribution.dist.name
                if dist_name == 'uniform':
                    entry["type"] = "uniform"
                    # For uniform(loc, scale), loc is args[0], scale is args[1]
                    entry["min_value"] = distribution.args[0]
                    entry["max_value"] = distribution.args[0] + distribution.args[1]
                elif dist_name == 'loguniform':
                    entry["type"] = "loguniform"
                    # loguniform(a, b) where a and b are min and max
                    entry["min_value"] = distribution.a
                    entry["max_value"] = distribution.b
                elif dist_name == 'randint':
                    entry["type"] = "randint"
                    # randint(low, high), high is exclusive
                    entry["min_value"] = distribution.a
                    entry["max_value"] = distribution.b - 1
                else:
                    entry["type"] = str(type(distribution))
                    entry["min_value"] = "N/A"
                    entry["max_value"] = "N/A"
            elif isinstance(distribution, list):
                entry["type"] = "categorical"
                entry["min_value"] = "N/A"
                entry["max_value"] = "N/A"
                entry["categories"] = str(distribution)
            else:
                entry["type"] = str(type(distribution))
                entry["min_value"] = "N/A"
                entry["max_value"] = "N/A"
            param_specs.append(entry)

df_search_spaces = pd.DataFrame(param_specs)

# Save to CSV
csv_path = Path(RESULTS_DIR) / "07_hyperparameter_search_spaces.csv"
df_search_spaces.to_csv(csv_path, index=False)
print(f"Malla de hiperparámetros guardada en: {csv_path}")
display(df_search_spaces)

---

## HPO por Experto con `HalvingRandomSearchCV`

Para cada experto (clúster), hacemos lo siguiente:

1. Filtrar los datos correspondientes al clúster.
2. Buscar la familia del modelo campeón desde el Notebook 05.
3. Ejecutar `HalvingRandomSearchCV` con el espacio de búsqueda apropiado.
4. Registrar los mejores parámetros, el mejor score y los resultados completos de validación cruzada.
5. Graficar una curva de convergencia que muestre cómo mejora el score a lo largo de las rondas de halving.

In [ ]:
# ── Estrategia de validación cruzada ────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=3, gap=0)  # gap = 14 días = horizonte de pronóstico

# ── Configuración de HPO ────────────────────────────────────────────────────
N_CANDIDATES   = 200
HALVING_FACTOR = 3
MIN_RESOURCES  = 100
SCORING        = "neg_root_mean_squared_error"

print(f"TimeSeriesSplit: {tscv.n_splits} particiones, gap={tscv.gap}")
print(f"HalvingRandomSearch: n_candidates={N_CANDIDATES}, factor={HALVING_FACTOR}")
print(f"  min_resources={MIN_RESOURCES}, scoring={SCORING}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Ejecutar HPO para cada experto
# ══════════════════════════════════════════════════════════════════════════
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

hpo_results = {}       # expert_id -> {best_params, best_score, cv_results, search_obj}
hpo_summary_rows = []  # para la tabla resumen final

for expert_id, model_name in sorted(best_models.items(), key=lambda x: int(x[0])):
    cid = int(expert_id)
    print(f"\n{'='*70}")
    print(f"Experto {expert_id} — Modelo: {model_name}")
    print(f"{'='*70}")

    # ── Filtrar datos del clúster ───────────────────────────────────────────
    mask = features_df["cluster_id"] == cid
    df_cluster = features_df.loc[mask].sort_index()
    print(f"  Clúster {cid}: {len(df_cluster):,} muestras")

    if len(df_cluster) < MIN_RESOURCES * 2:
        print(f"  ADVERTENCIA: Muy pocas muestras ({len(df_cluster)}). Reduciendo min_resources.")
        effective_min = max(50, len(df_cluster) // 10)
    else:
        effective_min = MIN_RESOURCES

    X = df_cluster[FEATURE_COLS].values
    y = df_cluster[TARGET].values

    # ── Recuperar espacio de búsqueda ─────────────────────────────────────────
    if model_name not in SEARCH_SPACES:
        print(f"  OMITIR: No se definió un espacio de búsqueda para '{model_name}'.")
        continue

    spec  = SEARCH_SPACES[model_name]
    model = clone(spec["model"])
    param_distributions = spec["params"]

    # ── HalvingRandomSearchCV ─────────────────────────────────────────
    search = HalvingRandomSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_candidates=N_CANDIDATES,
        factor=HALVING_FACTOR,
        resource="n_samples",
        min_resources=effective_min,
        cv=tscv,
        scoring=SCORING,
        random_state=RANDOM_SEED,
        n_jobs=1, # Cambiado de N_JOBS a 1 para depurar el problema de NaN
        verbose=0,
        refit=True,
    )

    t0 = time.time()
    search.fit(X, y)
    elapsed = time.time() - t0

    best_idx = search.best_index_
    best_rmse = -search.cv_results_["mean_test_score"][best_idx]

    # Calcular MAPE usando cross_val_score en el mejor modelo para tener ambos valores
    best_model = search.best_estimator_
    mape_scores = cross_val_score(best_model, X, y, cv=tscv, scoring="neg_mean_absolute_percentage_error")
    best_mape = -mape_scores.mean()

    print(f"  Mejor RMSE : {best_rmse:,.2f} kWh")
    print(f"  Mejor MAPE : {best_mape:.4%}")
    print(f"  Tiempo    : {elapsed:.1f} s")
    print(f"  Mejores parámetros:")
    for k, v in search.best_params_.items():
        if isinstance(v, float):
            print(f"    {k}: {v:.6f}")
        else:
            print(f"    {k}: {v}")

    # ── Guardar resultados ─────────────────────────────────────────────────
    hpo_results[expert_id] = {
        "model_name":  model_name,
        "best_params":  search.best_params_,
        "best_score":   best_rmse,
        "cv_results":   pd.DataFrame(search.cv_results_),
        "elapsed_s":    elapsed,
        "search_obj":   search,
    }

    hpo_summary_rows.append({
        "expert_id":  expert_id,
        "model":      model_name,
        "best_rmse":   best_rmse,
        "best_mape":   best_mape,
        "n_candidates_evaluated": len(search.cv_results_["params"]),
        "elapsed_s":  round(elapsed, 1),
        **{k: v for k, v in search.best_params_.items()},
    })

print(f"\nHPO completado para {len(hpo_results)} expertos.")

### 3.1  Curvas de Convergencia — Score vs. Ronda de Halving

Cada subgráfico muestra, para un experto, cómo evoluciona el **MAE promedio de prueba** a lo largo de las
sucesivas rondas de halving. Las rondas iniciales evalúan muchos candidatos con datos limitados;
las rondas posteriores evalúan menos candidatos con cantidades progresivamente mayores de datos.

In [ ]:
n_experts = len(hpo_results)
if n_experts == 0:
    print("No hay resultados de HPO para graficar.")
else:
    ncols = min(3, n_experts)
    nrows = int(np.ceil(n_experts / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows),
                              squeeze=False)

    for idx, (expert_id, res) in enumerate(sorted(hpo_results.items(),
                                                    key=lambda x: int(x[0]))):
        ax = axes[idx // ncols, idx % ncols]
        cv_df = res["cv_results"].copy()

        # HalvingRandomSearchCV guarda la columna 'iter' que indica la ronda de halving
        if "iter" in cv_df.columns:
            round_col = "iter"
        elif "n_resources" in cv_df.columns:
            round_col = "n_resources"
        else:
            round_col = None

        cv_df["rmse"] = -cv_df["mean_test_score"]

        if round_col is not None:
            # Agregación: media y desviación estándar del rmse por ronda de halving
            agg = cv_df.groupby(round_col)["rmse"].agg(["mean", "std", "count"]).reset_index()
            agg.columns = [round_col, "mean_rmse", "std_rmse", "n_cands"]

            ax.errorbar(agg[round_col], agg["mean_rmse"], yerr=agg["std_rmse"],
                        fmt="o-", color=PALETTE[0], capsize=4, linewidth=1.5,
                        markersize=6, label="RMSE promedio")

            # Anotar número de candidatos por ronda
            for _, row in agg.iterrows():
                ax.annotate(f"n={int(row['n_cands'])}",
                            (row[round_col], row["mean_rmse"]),
                            textcoords="offset points", xytext=(0, 12),
                            fontsize=8, ha="center", color=PALETTE[0])

            # Línea del mejor rmse
            ax.axhline(res["best_score"], ls="--", color=PALETTE[1], lw=1,
                       label=f"Mejor RMSE = {res['best_score']:,.1f}")

            xlabel = "Ronda de halving" if round_col == "iter" else "n_resources"
            ax.set_xlabel(xlabel)
        else:
            # Respaldo: graficar rmse ordenado para todos los candidatos
            sorted_rmse = cv_df["rmse"].sort_values().values
            ax.plot(range(len(sorted_rmse)), sorted_rmse, color=PALETTE[0], lw=1.5)
            ax.set_xlabel("Ranking de candidatos")

        ax.set_ylabel("RMSE (kWh)")
        ax.set_title(f"Experto {expert_id} — {res['model_name']}")
        ax.legend(loc="upper right", framealpha=0.9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Ocultar ejes no usados
    for idx in range(n_experts, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.suptitle("Convergencia del HPO — Successive Halving por Experto",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/07_hpo_convergence_curves.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Guardado: {RESULTS_DIR}/07_hpo_convergence_curves.png")

In [ ]:
n_experts = len(hpo_results)
if n_experts == 0:
    print("No HPO results to plot.")
else:
    ncols = min(3, n_experts)
    nrows = int(np.ceil(n_experts / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows),
                              squeeze=False)

    for idx, (expert_id, res) in enumerate(sorted(hpo_results.items(),
                                                    key=lambda x: int(x[0]))):
        ax = axes[idx // ncols, idx % ncols]
        cv_df = res["cv_results"].copy()

        if "iter" in cv_df.columns:
            round_col = "iter"
        elif "n_resources" in cv_df.columns:
            round_col = "n_resources"
        else:
            round_col = None

        cv_df["rmse"] = -cv_df["mean_test_score"]

        if round_col is not None:
            agg = cv_df.groupby(round_col)["rmse"].agg(["mean", "std", "count"]).reset_index()
            agg.columns = [round_col, "mean_rmse", "std_rmse", "n_cands"]

            ax.errorbar(agg[round_col], agg["mean_rmse"], yerr=agg["std_rmse"],
                        fmt="o-", color=PALETTE[0], capsize=4, linewidth=1.5,
                        markersize=6, label="Mean RMSE")

            for _, row in agg.iterrows():
                ax.annotate(f"n={int(row['n_cands'])}",
                            (row[round_col], row["mean_rmse"]),
                            textcoords="offset points", xytext=(0, 12),
                            fontsize=8, ha="center", color=PALETTE[0])

            ax.axhline(res["best_score"], ls="--", color=PALETTE[1], lw=1,
                       label=f"Best RMSE = {res['best_score']:,.1f}")

            xlabel = "Halving round" if round_col == "iter" else "n_resources"
            ax.set_xlabel(xlabel)
        else:
            sorted_rmse = cv_df["rmse"].sort_values().values
            ax.plot(range(len(sorted_rmse)), sorted_rmse, color=PALETTE[0], lw=1.5)
            ax.set_xlabel("Candidate rank")

        ax.set_ylabel("rmse (kWh)")
        ax.set_title(f"Expert {expert_id} — {res['model_name']}")
        ax.legend(loc="upper right", framealpha=0.9)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    for idx in range(n_experts, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.suptitle("HPO Convergence — Successive Halving per Expert",
                 fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/english/07_hpo_convergence_curves.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {RESULTS_DIR}/english/07_hpo_convergence_curves.png")

### 3.2  Tabla Resumen — Mejores Hiperparámetros por Experto

In [ ]:
hpo_summary_df = pd.DataFrame(hpo_summary_rows)
display_cols = ["expert_id", "model", "best_rmse", "best_mape", "n_candidates_evaluated", "elapsed_s"]
print("Resumen de HPO:")
display(hpo_summary_df[display_cols].style
        .format({"best_rmse": "{:,.2f}", "best_mape": "{:.2%}", "elapsed_s": "{:.1f}"})
        .set_caption("Mejor RMSE y MAPE por experto después de Halving Random Search")
        .set_table_styles([{"selector": "caption",
                            "props": [("font-weight", "bold"), ("font-size", "13px")]}]))

In [ ]:
# ── Tabla detallada de mejores parámetros ────────────────────────────────────────────
params_rows = []
for expert_id, res in sorted(hpo_results.items(), key=lambda x: int(x[0])):
    row = {"expert_id": expert_id, "model": res["model_name"]}
    for k, v in res["best_params"].items():
        # Limpiar nombre del parámetro (remover prefijo del pipeline)
        clean_k = k.split("__")[-1] if "__" in k else k
        row[clean_k] = round(v, 6) if isinstance(v, float) else v
    params_rows.append(row)

params_df = pd.DataFrame(params_rows)
print("\nHiperparámetros detallados por experto:")
display(params_df)

### 3.3  Distribución de Hiperparámetros — Gráfico de Coordenadas Paralelas

Para los expertos que usan la misma familia de modelos, mostramos un gráfico de coordenadas paralelas
de los 10 mejores candidatos para visualizar qué regiones del espacio de hiperparámetros son
seleccionadas de forma consistente.

In [ ]:
# Gráfico de coordenadas paralelas para cada familia de modelos presente en los resultados
from pandas.plotting import parallel_coordinates

families_in_use = set(res["model_name"] for res in hpo_results.values())

for family in sorted(families_in_use):
    # Recolectar los 10 mejores candidatos entre los expertos que usan esta familia
    top_candidates = []
    for expert_id, res in hpo_results.items():
        if res["model_name"] != family:
            continue
        cv_df = res["cv_results"].copy()
        cv_df["rmse"] = -cv_df["mean_test_score"]
        top10 = cv_df.nsmallest(10, "rmse")

        for _, row in top10.iterrows():
            p = row["params"]
            entry = {"expert": f"Experto {expert_id}"}
            for k, v in p.items():
                clean_k = k.split("__")[-1] if "__" in k else k
                if isinstance(v, (int, float)):
                    entry[clean_k] = v
            top_candidates.append(entry)

    if not top_candidates:
        continue

    pc_df = pd.DataFrame(top_candidates)
    numeric_cols = [c for c in pc_df.columns if c != "expert" and pc_df[c].dtype in ["float64", "int64"]]

    if len(numeric_cols) < 2:
        continue

    # Normalizar a [0, 1] para visualización
    pc_norm = pc_df.copy()
    for col in numeric_cols:
        cmin, cmax = pc_norm[col].min(), pc_norm[col].max()
        if cmax > cmin:
            pc_norm[col] = (pc_norm[col] - cmin) / (cmax - cmin)
        else:
            pc_norm[col] = 0.5

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 1.2), 4.5))
    parallel_coordinates(pc_norm[["expert"] + numeric_cols], "expert",
                         ax=ax, alpha=0.5, linewidth=1.5,
                         color=[PALETTE[i % len(PALETTE)] for i in range(pc_norm["expert"].nunique())])
    ax.set_title(f"Top-10 Configuraciones de HP — {family}", fontweight="bold")
    ax.set_ylabel("Valor normalizado")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/07_hpo_parallel_{family}.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Guardado: {RESULTS_DIR}/07_hpo_parallel_{family}.png")

In [ ]:
from pandas.plotting import parallel_coordinates

families_in_use = set(res["model_name"] for res in hpo_results.values())

for family in sorted(families_in_use):
    top_candidates = []
    for expert_id, res in hpo_results.items():
        if res["model_name"] != family:
            continue
        cv_df = res["cv_results"].copy()
        cv_df["mae"] = -cv_df["mean_test_score"]
        top10 = cv_df.nsmallest(10, "mae")

        for _, row in top10.iterrows():
            p = row["params"]
            entry = {"expert": f"Expert {expert_id}"}
            for k, v in p.items():
                clean_k = k.split("__")[-1] if "__" in k else k
                if isinstance(v, (int, float)):
                    entry[clean_k] = v
            top_candidates.append(entry)

    if not top_candidates:
        continue

    pc_df = pd.DataFrame(top_candidates)
    numeric_cols = [c for c in pc_df.columns if c != "expert" and pc_df[c].dtype in ["float64", "int64"]]

    if len(numeric_cols) < 2:
        continue
    pc_norm = pc_df.copy()
    for col in numeric_cols:
        cmin, cmax = pc_norm[col].min(), pc_norm[col].max()
        if cmax > cmin:
            pc_norm[col] = (pc_norm[col] - cmin) / (cmax - cmin)
        else:
            pc_norm[col] = 0.5

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 1.2), 4.5))
    parallel_coordinates(pc_norm[["expert"] + numeric_cols], "expert",
                         ax=ax, alpha=0.5, linewidth=1.5,
                         color=[PALETTE[i % len(PALETTE)] for i in range(pc_norm["expert"].nunique())])
    ax.set_title(f"Top-10 HP Configurations — {family}", fontweight="bold")
    ax.set_ylabel("Normalised value")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/07_hpo_parallel_{family}.png",
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {RESULTS_DIR}/english/07_hpo_parallel_{family}.png")

---

## HPO del Gating (Umbral y Temperatura)

La red de gating usa dos parámetros clave:

- **threshold** ($\theta$): nivel de confianza por encima del cual el gating asigna
  un enrutamiento duro a un único experto. Por debajo de este umbral, se usa una
  combinación suave (ponderada por distancia) entre expertos.
- **tau** ($\tau$): parámetro de temperatura para la ponderación basada en softmax de las distancias.
  Un $\tau$ más bajo produce distribuciones más agudas (más concentradas); un $\tau$ más alto
  produce una combinación más uniforme.

Como estos son solo dos parámetros con una grilla discreta pequeña, aquí es apropiada una
búsqueda exhaustiva en grilla (no hay necesidad de random search).

### Mecanismo de gating

Para una nueva entrada $\mathbf{x}$:

1. Calcular la distancia $d_k$ a cada centroide de clúster $k$.
2. Convertir a pesos softmax: $w_k = \frac{\exp(-d_k / \tau)}{\sum_j \exp(-d_j / \tau)}$
3. Si $\max_k(w_k) \geq \theta$: **enrutamiento duro** hacia $\arg\max_k(w_k)$.
4. En caso contrario: **enrutamiento suave** — promedio ponderado $\hat{y} = \sum_k w_k \, \hat{y}_k$
   (manteniendo solo los expertos con $w_k \geq w_{\min}$).

El MASE (Mean Absolute Scaled Error) en el conjunto de validación es la métrica global
para seleccionar la mejor combinación $(\theta, \tau)$.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.1  Preparar conjunto de validación y predicciones de expertos
# ══════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler

# Usar el ñltimo 20% de los datos como validación (partición temporal)
val_frac   = 0.20
n_total    = len(features_df)
n_val      = int(n_total * val_frac)
val_start  = n_total - n_val

df_val = features_df.iloc[val_start:].copy()
print(f"Conjunto de validación: {len(df_val):,} muestras ({val_frac:.0%} del total)")
print(f"  Período: {df_val.index.min()} to {df_val.index.max()}")

# ── Escalar variables para calcular distancias sensatas al gating ────────
scaler_gating = StandardScaler()
df_train = features_df.iloc[:val_start].copy()

# Entrenar scaler con train y transformar train y val
df_train_scaled = df_train.copy()
df_train_scaled[FEATURE_COLS] = scaler_gating.fit_transform(df_train[FEATURE_COLS])

df_val_scaled = df_val.copy()
df_val_scaled[FEATURE_COLS] = scaler_gating.transform(df_val[FEATURE_COLS])

# ── Calcular centroides de clúster (vector promedio de variables por clúster en escala) ────────
centroids = {}
for cid in sorted(features_df["cluster_id"].unique()):
    mask_c = df_train["cluster_id"] == cid
    centroids[cid] = df_train_scaled.loc[mask_c, FEATURE_COLS].mean().values

centroid_ids = sorted(centroids.keys())
centroid_matrix = np.vstack([centroids[c] for c in centroid_ids])  # (K, D)

# ── Generar predicciones por experto sobre el conjunto de validación ──────────────────
expert_preds_val = {}  # expert_id -> np.array de predicciones

for expert_id, res in hpo_results.items():
    best_estimator = res["search_obj"].best_estimator_
    # Los modelos esperan las variables originales (sin el escalado del gating)
    # ya que cada modelo experto tiene su propio Pipeline con un StandardScaler interno.
    X_val_original = df_val[FEATURE_COLS].values
    expert_preds_val[expert_id] = best_estimator.predict(X_val_original)

y_val = df_val[TARGET].values
print(f"\nSe generaron predicciones de {len(expert_preds_val)} expertos sobre el conjunto de validación.")


In [ ]:
expert_preds_val

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.2  Función de evaluación del gating
# ══════════════════════════════════════════════════════════════════════════

def compute_gating_predictions(
    X_val: np.ndarray,
    expert_preds: dict,
    centroid_matrix: np.ndarray,
    centroid_ids: list,
    threshold: float,
    tau: float,
    min_prob: float = 0.05,
) -> np.ndarray:
    """
    Aplica gating adaptativo para combinar las predicciones de los expertos.

    Parameters
    ----------
    X_val : matriz de variables de validación (N, D)
    expert_preds : dict expert_id -> predicciones (N,)
    centroid_matrix : centroides de clúster (K, D)
    centroid_ids : lista de ids de clúster que coinciden con las filas de centroid_matrix
    threshold : umbral de confianza para enrutamiento duro
    tau : temperatura del softmax
    min_prob : peso mínimo para incluir un experto en el enrutamiento suave

    Returns
    -------
    y_pred : predicciones combinadas (N,)
    """
    N = X_val.shape[0]
    y_pred = np.zeros(N)

    # Calcular distancias a los centroides: (N, K)
    from scipy.spatial.distance import cdist
    dists = cdist(X_val, centroid_matrix, metric="euclidean")

    for i in range(N):
        # Pesos softmax con temperatura
        d_i = dists[i]  # (K,)
        log_weights = -d_i / (tau + 1e-12)
        log_weights -= log_weights.max()  # estabilidad numérica
        weights = np.exp(log_weights)
        weights /= weights.sum()

        if weights.max() >= threshold:
            # Enrutamiento duro
            best_k = centroid_ids[np.argmax(weights)]

            # Manejar correctamente llaves enteras o strings en el diccionario
            if best_k in expert_preds:
                y_pred[i] = expert_preds[best_k][i]
            elif str(best_k) in expert_preds:
                y_pred[i] = expert_preds[str(best_k)][i]
            else:
                # Respaldo: promedio ponderado de los expertos disponibles
                total_w = 0.0
                for j, cid in enumerate(centroid_ids):
                    if cid in expert_preds:
                        y_pred[i] += weights[j] * expert_preds[cid][i]
                        total_w += weights[j]
                    elif str(cid) in expert_preds:
                        y_pred[i] += weights[j] * expert_preds[str(cid)][i]
                        total_w += weights[j]
                if total_w > 0:
                    y_pred[i] /= total_w
        else:
            # Enrutamiento suave — combinar expertos por encima de min_prob
            total_w = 0.0
            for j, cid in enumerate(centroid_ids):
                if weights[j] < min_prob:
                    continue
                if cid in expert_preds:
                    y_pred[i] += weights[j] * expert_preds[cid][i]
                    total_w += weights[j]
                elif str(cid) in expert_preds:
                    y_pred[i] += weights[j] * expert_preds[str(cid)][i]
                    total_w += weights[j]
            if total_w > 0:
                y_pred[i] /= total_w

    return y_pred


def mase(y_true: np.ndarray, y_pred: np.ndarray, seasonal_period: int = 24) -> float:
    """
    Mean Absolute Scaled Error.

    Escalado por el MAE dentro de muestra de un pronóstico ingenuo estacional (período = 24h).
    """
    mae_pred = np.mean(np.abs(y_true - y_pred))
    # Error ingenuo estacional
    naive_errors = np.abs(y_true[seasonal_period:] - y_true[:-seasonal_period])
    mae_naive = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    return mae_pred / mae_naive if mae_naive > 0 else np.inf


print("Funciones de evaluación del gating definidas.")

In [ ]:
X_val_feat

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4.3  Busqueda en grilla sobre (threshold, tau)
# ══════════════════════════════════════════════════════════════════════════
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

THRESHOLDS = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]
TAUS       = [0.1, 0.5, 1.0, 2.0, 5.0]
MIN_PROB   = 0.05

# Usar las variables ESCALADAS para calcular la distancia en el gating
X_val_feat = df_val_scaled[FEATURE_COLS].values

gating_grid_rows = []

print(f"Evaluando {len(THRESHOLDS) * len(TAUS)} combinaciones de (threshold, tau)...")

for thr in THRESHOLDS:
    for tau_val in TAUS:
        y_pred_gated = compute_gating_predictions(
            X_val=X_val_feat,
            expert_preds=expert_preds_val,
            centroid_matrix=centroid_matrix,
            centroid_ids=centroid_ids,
            threshold=thr,
            tau=tau_val,
            min_prob=MIN_PROB,
        )
        mae_val  = mean_absolute_error(y_val, y_pred_gated)
        rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_gated))
        mape_val = mean_absolute_percentage_error(y_val, y_pred_gated)
        mase_val = mase(y_val, y_pred_gated, seasonal_period=24)

        gating_grid_rows.append({
            "threshold": thr,
            "tau":       tau_val,
            "MAE":       mae_val,
            "RMSE":      rmse_val,
            "MAPE":      mape_val,
            "MASE":      mase_val,
        })

gating_grid_df = pd.DataFrame(gating_grid_rows)

# ── Mejor combinación ──────────────────────────────────────────────────────
best_gating_idx = gating_grid_df["MASE"].idxmin()
best_gating     = gating_grid_df.loc[best_gating_idx]

print(f"\nMejor configuración de gating:")
print(f"  threshold = {best_gating['threshold']}")
print(f"  tau       = {best_gating['tau']}")
print(f"  MAE       = {best_gating['MAE']:,.2f} kWh")
print(f"  RMSE      = {best_gating['RMSE']:,.2f} kWh")
print(f"  MAPE      = {best_gating['MAPE']:.4f} %")
print(f"  MASE      = {best_gating['MASE']:.4f}")


In [ ]:
# ── 4.4  Mapa de calor: MASE sobre la grilla (threshold, tau) ────────────────────────
pivot_mase = gating_grid_df.pivot(index="threshold", columns="tau", values="MASE")
pivot_mape  = gating_grid_df.pivot(index="threshold", columns="tau", values="MAPE")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Mapa de calor de MASE
sns.heatmap(pivot_mase, annot=True, fmt=".4f", cmap="YlOrRd",
            ax=axes[0], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MASE"})
axes[0].set_title("MASE — Búsqueda en Grilla del Gating", fontweight="bold")
axes[0].set_xlabel(r"Temperatura $\tau$")
axes[0].set_ylabel(r"Umbral $\theta$")

# Resaltar la mejor celda
best_row = list(pivot_mase.index).index(best_gating["threshold"])
best_col = list(pivot_mase.columns).index(best_gating["tau"])
axes[0].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

# Mapa de calor de mape
sns.heatmap(pivot_mape, annot=True, fmt=",.4f", cmap="YlOrRd",
            ax=axes[1], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MAPE %"})
axes[1].set_title("MAPE — Búsqueda en Grilla del Gating", fontweight="bold")
axes[1].set_xlabel(r"Temperatura $\tau$")
axes[1].set_ylabel(r"Umbral $\theta$")

# Resaltar la mejor celda
axes[1].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

fig.suptitle("Optimización de Hiperparámetros del Gating — Búsqueda en Grilla",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/07_hpo_gating_heatmap.png",
            bbox_inches="tight", facecolor="white")
plt.show()
print(f"Guardado: {RESULTS_DIR}/07_hpo_gating_heatmap.png")

In [ ]:
pivot_mase = gating_grid_df.pivot(index="threshold", columns="tau", values="MASE")
pivot_mape  = gating_grid_df.pivot(index="threshold", columns="tau", values="MAPE")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# MASE heatmap
sns.heatmap(pivot_mase, annot=True, fmt=".4f", cmap="YlOrRd",
            ax=axes[0], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MASE"})
axes[0].set_title("MASE — Gating Grid Search", fontweight="bold")
axes[0].set_xlabel(r"Temperature $\tau$")
axes[0].set_ylabel(r"Threshold $\theta$")

best_row = list(pivot_mase.index).index(best_gating["threshold"])
best_col = list(pivot_mase.columns).index(best_gating["tau"])
axes[0].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

# MAE heatmap
sns.heatmap(pivot_mape, annot=True, fmt=",.1f", cmap="YlOrRd",
            ax=axes[1], linewidths=0.5, linecolor="white",
            cbar_kws={"label": "MAPE %"})
axes[1].set_title("MAPE — Gating Grid Search", fontweight="bold")
axes[1].set_xlabel(r"Temperature $\tau$")
axes[1].set_ylabel(r"Threshold $\theta$")

axes[1].add_patch(plt.Rectangle((best_col, best_row), 1, 1,
                                 fill=False, edgecolor="black", linewidth=3))

fig.suptitle("Gating Hyperparameter Optimisation — Grid Search",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(f"{RESULTS_DIR}/english/07_hpo_gating_heatmap.png",
            bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {RESULTS_DIR}/english/07_hpo_gating_heatmap.png")

In [ ]:
# ── 4.5  Tabla completa de la grilla de gating ──────────────────────────────────────────
print("\nResultados completos de la grilla de gating (ordenados por MASE):")
display(
    gating_grid_df
    .sort_values("MASE")
    .reset_index(drop=True)
    .style
    .format({"MAE": "{:,.2f}", "MASE": "{:.4f}", "MAPE": "{:.2f}", "RMSE": "{:,.2f}"})
    .highlight_min(subset=["MASE"], color="#d4edda")
    .set_caption("Resultados de la búsqueda en grilla del gating")
)

---

## Persistir Archivos Optimizados

Guardamos:

1. **Mejores hiperparámetros** por experto en formato JSON.
2. **Modelos optimizados** (refitted `best_estimator_`) como archivos pickle.
3. **Tabla de resultados de HPO** como CSV.
4. **Parámetros óptimos del gating** como JSON.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5.1  Guardar mejores hiperparámetros (JSON)
# ══════════════════════════════════════════════════════════════════════════
experts_dir = Path(MODELS_DIR) / "hyperparameters"
experts_dir.mkdir(parents=True, exist_ok=True)

best_params_dict = {}
for expert_id, res in hpo_results.items():
    # Convertir tipos de numpy para serialización en JSON
    params_clean = {}
    for k, v in res["best_params"].items():
        if isinstance(v, (np.integer,)):
            params_clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            params_clean[k] = float(v)
        else:
            params_clean[k] = v
    best_params_dict[expert_id] = {
        "model": res["model_name"],
        "best_MAE": float(res["best_score"]),
        "params": params_clean,
    }

hp_path = experts_dir / "best_hyperparameters.json"
with open(hp_path, "w") as f:
    json.dump(best_params_dict, f, indent=2)
print(f"Mejores hiperparámetros guardados: {hp_path}")

# ══════════════════════════════════════════════════════════════════════════
# 5.2  Guardar modelos optimizados (pickle)
# ══════════════════════════════════════════════════════════════════════════

for expert_id, res in hpo_results.items():
    model_path = experts_dir / f"expert_{expert_id}_optimised.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(res["search_obj"].best_estimator_, f)
    print(f"  Guardado: {model_path}")

# ══════════════════════════════════════════════════════════════════════════
# 5.3  Guardar tabla de resultados de HPO (CSV)
# ══════════════════════════════════════════════════════════════════════════

hpo_csv_path = Path(RESULTS_DIR) / "07_hpo_results.csv"
hpo_summary_df.to_csv(hpo_csv_path, index=False)
print(f"\nResumen de HPO guardado: {hpo_csv_path}")

# ══════════════════════════════════════════════════════════════════════════
# 5.4  Guardar parámetros óptimos del gating (JSON)
# ══════════════════════════════════════════════════════════════════════════

gating_best = {
    "threshold": float(best_gating["threshold"]),
    "tau":       float(best_gating["tau"]),
    "min_prob":  MIN_PROB,
    "MAE":       float(best_gating["MAE"]),
    "MASE":      float(best_gating["MASE"]),
    "MAPE":      float(best_gating["MAPE"]),
    "RMSE":      float(best_gating["RMSE"])
}

gating_path = experts_dir / "best_gating_params.json"
with open(gating_path, "w") as f:
    json.dump(gating_best, f, indent=2)
print(f"Mejores parámetros de gating guardados: {gating_path}")

# ── Guardar grilla completa de gating ─────────────────────────────────────────────────
gating_csv_path = Path(RESULTS_DIR) / "07_gating_grid_results.csv"
gating_grid_df.to_csv(gating_csv_path, index=False)
print(f"Grilla de gating guardada: {gating_csv_path}")